# 论文 17：变分有损自编码器
## Xi Chen, Diederik P. Kingma et al.（2016）

### VAE：具有学习潜在空间的生成模型

将深度学习与生成建模的变分推理相结合。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 变分自编码器（VAE）基础

VAE 学习以下两个分布：
- **编码器**：q(z|x) - 近似后验
- **解码器**：p(x|z) - 生成模型

**目标函数**：ELBO 由重建项和 KL 散度项组成。

In [ ]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class VAE:
    def __init__(self, input_dim, hidden_dim, latent_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        # 编码器：x -> h -> (mu, log_var)
        self.W_enc_h = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b_enc_h = np.zeros(hidden_dim)
        
        self.W_mu = np.random.randn(hidden_dim, latent_dim) * 0.1
        self.b_mu = np.zeros(latent_dim)
        
        self.W_logvar = np.random.randn(hidden_dim, latent_dim) * 0.1
        self.b_logvar = np.zeros(latent_dim)
        
        # 解码器：z -> h -> x_recon
        self.W_dec_h = np.random.randn(latent_dim, hidden_dim) * 0.1
        self.b_dec_h = np.zeros(hidden_dim)
        
        self.W_recon = np.random.randn(hidden_dim, input_dim) * 0.1
        self.b_recon = np.zeros(input_dim)
    
    def encode(self, x):
        """将输入编码为潜在分布参数
        
        返回：q(z|x) 的 mu、log_var"""
        h = relu(np.dot(x, self.W_enc_h) + self.b_enc_h)
        mu = np.dot(h, self.W_mu) + self.b_mu
        log_var = np.dot(h, self.W_logvar) + self.b_logvar
        return mu, log_var
    
    def reparameterize(self, mu, log_var):
        """重参数化 trick：z = mu + sigma * epsilon
        其中 epsilon ~ N(0, I)"""
        std = np.exp(0.5 * log_var)
        epsilon = np.random.randn(*mu.shape)
        z = mu + std * epsilon
        return z
    
    def decode(self, z):
        """解码潜在代码以重建
        
        返回：重建x"""
        h = relu(np.dot(z, self.W_dec_h) + self.b_dec_h)
        x_recon = sigmoid(np.dot(h, self.W_recon) + self.b_recon)
        return x_recon
    
    def forward(self, x):
        '完整前向传播'
        # 编码
        mu, log_var = self.encode(x)
        
        # 采样潜变量
        z = self.reparameterize(mu, log_var)
        
        # 解码
        x_recon = self.decode(z)
        
        return x_recon, mu, log_var, z
    
    def loss(self, x, x_recon, mu, log_var):
        'VAE 损失 = 重建损失 + KL 散度'
        # 重建损失（二元交叉熵）
        recon_loss = -np.sum(
            x * np.log(x_recon + 1e-8) + 
            (1 - x) * np.log(1 - x_recon + 1e-8)
        )
        
        # KL 散度：KL(q(z|x) || p(z))
        # 其中 p(z) = N(0, I)
        # KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
        kl_loss = -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))
        
        return recon_loss + kl_loss, recon_loss, kl_loss

# 创建VAE
input_dim = 16  # e.g.，4x4 图像展平
hidden_dim = 32
latent_dim = 2  # 2D 可视化

vae = VAE(input_dim, hidden_dim, latent_dim)
print(f"VAE created:")
print(f"  Input: {input_dim}")
print(f"  Hidden: {hidden_dim}")
print(f"  Latent: {latent_dim}")

## 生成合成数据

用于演示的简单 4x4 模式

In [ ]:
def generate_patterns(num_samples=100):
    '生成简单的 4x4 二进制模式'
    data = []
    
    for i in range(num_samples):
        pattern = np.zeros((4, 4))
        
        if i % 4 == 0:
            # 水平线
            pattern[1:2, :] = 1
        elif i % 4 == 1:
            # 垂线
            pattern[:, 2:3] = 1
        elif i % 4 == 2:
            # 对角线
            np.fill_diagonal(pattern, 1)
        else:
            # 角广场
            pattern[:2, :2] = 1
        
        # 添加小噪音
        noise = np.random.randn(4, 4) * 0.05
        pattern = np.clip(pattern + noise, 0, 1)
        
        data.append(pattern.flatten())
    
    return np.array(data)

# 生成训练数据
X_train = generate_patterns(200)

# 可视化样本
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(4, 4), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Pattern {i}')
    ax.axis('off')
plt.suptitle('Training Data Samples')
plt.show()

print(f"Generated {len(X_train)} training samples")

## 测试前向传播与损失

In [ ]:
# 在单个示例上进行测试
x = X_train[0:1]
x_recon, mu, log_var, z = vae.forward(x)

total_loss, recon_loss, kl_loss = vae.loss(x, x_recon, mu, log_var)

print(f"Forward pass:")
print(f"  Input shape: {x.shape}")
print(f"  Latent mu: {mu}")
print(f"  Latent log_var: {log_var}")
print(f"  Latent z: {z}")
print(f"  Reconstruction shape: {x_recon.shape}")
print(f"\nLosses:")
print(f"  Total: {total_loss:.4f}")
print(f"  Reconstruction: {recon_loss:.4f}")
print(f"  KL Divergence: {kl_loss:.4f}")

# 可视化重建
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(x.reshape(4, 4), cmap='gray', vmin=0, vmax=1)
ax1.set_title('Original')
ax1.axis('off')

ax2.imshow(x_recon.reshape(4, 4), cmap='gray', vmin=0, vmax=1)
ax2.set_title('Reconstruction (Untrained)')
ax2.axis('off')

plt.show()

## 可视化潜在空间

由于 latent_dim=2，可以直接可视化模型学习到的潜在表示。

In [ ]:
# 对所有训练数据进行编码
latent_codes = []
pattern_types = []

for i, x in enumerate(X_train):
    mu, log_var = vae.encode(x.reshape(1, -1))
    latent_codes.append(mu[0])
    pattern_types.append(i % 4)

latent_codes = np.array(latent_codes)
pattern_types = np.array(pattern_types)

# 绘制潜在空间
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    latent_codes[:, 0], 
    latent_codes[:, 1], 
    c=pattern_types, 
    cmap='tab10', 
    alpha=0.6,
    s=50
)
plt.colorbar(scatter, label='Pattern Type')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('Latent Space (Untrained VAE)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Latent space visualization shows distribution of encoded patterns")

## 从先验分布采样并生成数据

从 z ~ N(0, I) 中采样，再通过解码器生成新样本。

In [ ]:
# 标准正常先验样本
num_samples = 8
z_samples = np.random.randn(num_samples, latent_dim)

# 生成样本
generated = []
for z in z_samples:
    x_gen = vae.decode(z.reshape(1, -1))
    generated.append(x_gen[0])

# 可视化生成的样本
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.imshow(generated[i].reshape(4, 4), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'z={z_samples[i][:2]}')
    ax.axis('off')

plt.suptitle('Generated Samples from Prior p(z) = N(0, I)', fontsize=14)
plt.tight_layout()
plt.show()

## 潜在空间中的插值

在潜在空间中的两点之间平滑插值

In [ ]:
# 编码两种不同的模式
x1 = X_train[0:1]  # 图案类型 0
x2 = X_train[1:2]  # 图案类型1

mu1, _ = vae.encode(x1)
mu2, _ = vae.encode(x2)

# 插
num_steps = 8
interpolated = []

for alpha in np.linspace(0, 1, num_steps):
    z_interp = (1 - alpha) * mu1 + alpha * mu2
    x_interp = vae.decode(z_interp)
    interpolated.append(x_interp[0])

# 可视化插值
fig, axes = plt.subplots(1, num_steps, figsize=(16, 2))

for i, ax in enumerate(axes):
    ax.imshow(interpolated[i].reshape(4, 4), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'α={i/(num_steps-1):.2f}')
    ax.axis('off')

plt.suptitle('Latent Space Interpolation', fontsize=14, y=1.1)
plt.tight_layout()
plt.show()

print("Smooth transitions show continuity in latent space")

## 重参数化技巧可视化

In [ ]:
# 显示来自同一分布的多个样本
x = X_train[0:1]
mu, log_var = vae.encode(x)

# 多次采样
num_samples = 100
z_samples = []
for _ in range(num_samples):
    z = vae.reparameterize(mu, log_var)
    z_samples.append(z[0])

z_samples = np.array(z_samples)

# 地块分布
plt.figure(figsize=(10, 8))
plt.scatter(z_samples[:, 0], z_samples[:, 1], alpha=0.3, s=20)
plt.scatter(mu[0, 0], mu[0, 1], color='red', s=200, marker='*', label='μ', zorder=5)

# 绘制 2 个标准差的椭圆
std = np.exp(0.5 * log_var[0])
theta = np.linspace(0, 2*np.pi, 100)
ellipse_x = mu[0, 0] + 2 * std[0] * np.cos(theta)
ellipse_y = mu[0, 1] + 2 * std[1] * np.sin(theta)
plt.plot(ellipse_x, ellipse_y, 'r--', label='2σ boundary', linewidth=2)

plt.xlabel('z₁')
plt.ylabel('z₂')
plt.title('Reparameterization Trick: z = μ + σ ⊙ ε, where ε ~ N(0,I)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

print(f"μ = {mu[0]}")
print(f"σ = {std}")
print(f"Sample mean: {z_samples.mean(axis=0)}")
print(f"Sample std: {z_samples.std(axis=0)}")

## 要点

### VAE 架构：
1. **编码器**：q_φ(z|x) - 将输入映射到潜在分布
2. **重参数化**：z = μ + σ ⊙ ε（启用反向传播）
3. **解码器**：p_θ(x|z) - 从潜变量生成输出

### 损失函数（ELBO）：
```
L = E[log p(x|z)] - KL(q(z|x) || p(z))
  = Reconstruction Loss - KL Divergence
```

### KL 散度：
- 正则化潜在空间以接近先验 p(z) = N(0, I)
- 防止过拟合
- 确保平滑的潜在空间

### 重参数化技巧
- 使采样可微分
- z = μ(x) + σ(x) ⊙ ε，其中 ε ~ N(0, I)
- 梯度流过 μ 和 σ

### 主要特点
- **生成**：可以对新数据进行采样
- **连续潜在空间**：平滑插值
- **概率建模**：能够表示不确定性
- **解耦表示**：可以通过 β-VAE 等变体学习

### 应用：
- 图像生成
- 降维
- 半监督学习
- 异常检测
- 数据增强

### 变体
- **β-VAE**：使用加权 KL 项促进表示解耦
- **条件 VAE**：进行条件生成
- **分层 VAE**：使用多个潜在层级
- **VQ-VAE**：使用离散潜变量